# Proyecto de Machine Learning
Este notebook contiene el c?digo para un proyecto de Machine Learning que utiliza un dataset de salud para predecir accidentes cerebrovasculares.

## Importaci?n de Librer?as



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Carga de Datos e Inspecci?n

En esta secci?n, cargaremos el dataset y realizaremos una inspecci?n inicial para entender mejor su estructura y contenido.

El dataset utilizado en este proyecto fue obtenido de Kaggle y se puede encontrar en el siguiente enlace: [Stroke Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset/data).

### Descripci?n del Dataset

El dataset contiene informaci?n sobre pacientes y se utiliza para predecir la probabilidad de que un paciente sufra un accidente cerebrovascular. Los accidentes cerebrovasculares son una de las principales causas de muerte y discapacidad en todo el mundo. Este dataset incluye las siguientes caracter?sticas:

- `id`: Identificador ?nico del paciente.
- `gender`: G?nero del paciente.
- `age`: Edad del paciente.
- `hypertension`: 0 si el paciente no tiene hipertensi?n, 1 si el paciente tiene hipertensi?n.
- `heart_disease`: 0 si el paciente no tiene enfermedades del coraz?n, 1 si el paciente tiene enfermedades del coraz?n.
- `ever_married`: Si el paciente alguna vez estuvo casado.
- `work_type`: Tipo de trabajo del paciente.
- `Residence_type`: Tipo de residencia del paciente (urbana o rural).
- `avg_glucose_level`: Nivel promedio de glucosa en sangre.
- `bmi`: ?ndice de masa corporal.
- `smoking_status`: Estado de fumador del paciente.
- `stroke`: 1 si el paciente tuvo un accidente cerebrovascular, 0 en caso contrario.

A continuaci?n, cargaremos el dataset y realizaremos una inspecci?n inicial.

In [ ]:
df=pd.read_csv(r"healthcare-dataset-stroke-data.csv")

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.info()


In [ ]:
df.head()

In [ ]:
df["gender"].value_counts()

In [ ]:
# Filtrar el DataFrame eliminando las filas donde gender es 'Other' ya que habia un solo registro con ese valor
df = df[df['gender'] != 'Other']

In [ ]:
df["gender"].value_counts()

### Manejo de Valores Nulos

Durante la inspecci?n inicial, observamos que la columna `bmi` contiene valores nulos. Para manejar estos valores nulos, decidimos rellenarlos con la mediana de la columna `bmi`. La mediana es una mejor opci?n que la media en este caso, ya que la columna `bmi` contiene muchos valores at?picos (outliers) que pueden sesgar la media.

A continuaci?n, mostramos los gr?ficos que justifican esta decisi?n.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df['bmi'], kde=True)
plt.title("Distribuci?n de BMI")
plt.show()

sns.boxplot(df['bmi'])
plt.title("Boxplot de BMI")
plt.show()

In [ ]:
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

In [ ]:
#Se elimina la columna ID ya que no aporta informaci?n
df = df.drop(columns=['id'])

In [ ]:
# Separar columnas categ?ricas y num?ricas
# En esta secci?n, separamos las columnas categ?ricas y num?ricas.
# Las columnas 'heart_disease' e 'hypertension' no se consideran categ?ricas porque son booleanas.
categorical = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
numerical = ['age', 'avg_glucose_level', 'bmi']

#value counts de las columnas seleccionada en la variable categorical
for col in categorical:
    print(df[col].value_counts())
    print()

categorical_sin_booleanas = ['work_type', 'Residence_type', 'smoking_status']

#Gender rename para female=0 y para male =1
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

#ever married para Yes = 1 y para No = 0   
df['ever_married'] = df['ever_married'].map({'Yes': 1, 'No': 0})

# Aplicar one-hot encoding solo a las columnas categ?ricas
# Utilizamos one-hot encoding para convertir las variables categ?ricas en variables dummy.
# Esto es necesario para que los algoritmos de machine learning puedan procesar estas variables.
df = pd.get_dummies(df, columns=categorical_sin_booleanas, drop_first=True)



## Divisi?n de Datos: Train-Test Split

En esta secci?n, dividiremos los datos en conjuntos de entrenamiento y prueba.

Primero, separamos las caracter?sticas (features) de la variable objetivo (target). La variable objetivo en este caso es `stroke`, que indica si el paciente ha tenido un accidente cerebrovascular.

A continuaci?n, utilizamos la funci?n `train_test_split` de `sklearn` para dividir los datos en conjuntos de entrenamiento y prueba. Usamos un 80% de los datos para el entrenamiento y un 20% para la prueba. Adem?s, utilizamos el par?metro `stratify` para asegurar que la proporci?n de clases en los conjuntos de entrenamiento y prueba sea similar a la del dataset original.

In [ ]:
# Mostrar las columnas del DataFrame
df.columns

In [ ]:
# Separar las caracter?sticas (features) de la variable objetivo (target)
# La variable objetivo es 'stroke'
features = df.drop(columns=["stroke"])
target = df["stroke"]

# Dividir los datos en conjuntos de entrenamiento y prueba
# Usamos un 80% de los datos para el entrenamiento y un 20% para la prueba
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)

## Manejo del Desbalanceo de Clases: Oversampling

En esta secci?n, abordaremos el desbalanceo de clases en la variable objetivo `stroke`. El desbalanceo de clases puede afectar negativamente el rendimiento del modelo, ya que los modelos de Machine Learning tienden a estar sesgados hacia la clase mayoritaria.

Primero, mostramos el desbalanceo de clases en el conjunto de entrenamiento.



In [ ]:

# Mostrar el desbalanceo de clases en el conjunto de entrenamiento
print(y_train.value_counts())

Para manejar este desbalanceo, utilizamos la t?cnica de oversampling. Esta t?cnica consiste en aumentar el n?mero de muestras de la clase minoritaria replicando aleatoriamente sus ejemplos hasta que ambas clases tengan el mismo n?mero de muestras.

A continuaci?n, realizamos el oversampling de la clase minoritaria.



In [ ]:
# Agregar la columna 'stroke' a X_train para facilitar el oversampling
X_train["stroke"] = y_train

# Separar las clases mayoritaria y minoritaria
stroke1 = X_train[X_train["stroke"] == 1]
stroke0 = X_train[X_train["stroke"] == 0]

# Mostrar el tama?o de cada clase antes del oversampling
print("Clase 1 (stroke):", stroke1.shape)
print("Clase 0 (no stroke):", stroke0.shape)

# Realizar el oversampling de la clase minoritaria
from sklearn.utils import resample

stroke1_oversampling = resample(stroke1, 
                                replace=True, 
                                n_samples=len(stroke0), 
                                random_state=0)

# Combinar las clases mayoritaria y minoritaria despu?s del oversampling
X_train = pd.concat([stroke1_oversampling, stroke0])

# Separar las caracter?sticas (features) de la variable objetivo (target) despu?s del oversampling
y_train = X_train["stroke"]
X_train = X_train.drop(columns=["stroke"])

# Mostrar el tama?o de la clase minoritaria despu?s del oversampling
print("Tama?o de la clase minoritaria despu?s del oversampling:", y_train.value_counts())

## Scaling

En esta secci?n, escalaremos las caracter?sticas utilizando `StandardScaler`. Esto es importante para los modelos de regresi?n log?stica, ya que asegura que todas las caracter?sticas contribuyan de manera similar al modelo.

El escalado transforma las caracter?sticas para que tengan una media de 0 y una desviaci?n est?ndar de 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Inicializar el escalador
scaler = StandardScaler()

# Ajustar y transformar los datos de entrenamiento
X_train = scaler.fit_transform(X_train)

# Transformar los datos de prueba
X_test = scaler.transform(X_test)

## Guardar el Scaler

Para asegurar la reproducibilidad y facilitar el despliegue del modelo, guardaremos el escalador en un archivo `.pkl`. Esto nos permitir? reutilizar el mismo escalador para transformar nuevos datos de manera consistente.

In [ ]:
import joblib

# Guardar el escalador en un archivo .pkl
joblib.dump(scaler, "scaler.pkl")

## Entrenamiento y Evaluaci?n del Modelo K-Nearest Neighbors (KNN)

En esta secci?n, entrenaremos un modelo K-Nearest Neighbors (KNN) para predecir la variable objetivo `stroke`.

KNN es sensible a los outliers, lo que puede afectar negativamente su rendimiento. Por esta raz?n, es importante escalar las caracter?sticas antes de entrenar el modelo.

A continuaci?n, entrenamos el modelo KNN utilizando los datos escalados y evaluamos su rendimiento.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint

# Crear el modelo KNN
knn = KNeighborsClassifier()

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'n_neighbors': randint(low=1, high=50),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_knn_oversampling = RandomizedSearchCV(
    estimator=knn,                   # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento balanceados con oversampling
random_search_knn_oversampling.fit(X_train, y_train)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_knn_oversampling.best_params_)

# Entrenar el modelo KNN con los mejores par?metros
best_knn_oversampling = random_search_knn_oversampling.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba
y_pred_knn_oversampling = best_knn_oversampling.predict(X_test)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_test, y_pred_knn_oversampling))
print("\nClassification Report:\n", classification_report(y_test, y_pred_knn_oversampling))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_knn_oversampling))



## An?lisis de Resultados - K-Nearest Neighbors (KNN) con Oversampling

El modelo K-Nearest Neighbors (KNN) con oversampling obtuvo una precisi?n general de 91%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.11. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo de Regresi?n Log?stica con Oversampling

En esta secci?n, entrenaremos y evaluaremos un modelo de Regresi?n Log?stica utilizando los datos balanceados con oversampling. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import uniform

# Crear el modelo de Regresi?n Log?stica
log_reg = LogisticRegression(solver='liblinear', random_state=42)

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'C': uniform(loc=0, scale=4),  # Regularizaci?n
    'penalty': ['l1', 'l2']        # Tipo de penalizaci?n
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_log_reg_oversampling = RandomizedSearchCV(
    estimator=log_reg,               # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento balanceados con oversampling
random_search_log_reg_oversampling.fit(X_train, y_train)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_log_reg_oversampling.best_params_)

# Entrenar el modelo de Regresi?n Log?stica con los mejores par?metros
best_log_reg_oversampling = random_search_log_reg_oversampling.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba
y_pred_log_reg_oversampling = best_log_reg_oversampling.predict(X_test)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_test, y_pred_log_reg_oversampling))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log_reg_oversampling))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log_reg_oversampling))

## An?lisis de Resultados - Regresi?n Log?stica con Oversampling

El modelo de Regresi?n Log?stica con oversampling obtuvo una precisi?n general de 72.11%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.22. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo Random Forest

En esta secci?n, entrenaremos un modelo Random Forest para predecir la variable objetivo `stroke`.

Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

A continuaci?n, entrenamos el modelo Random Forest utilizando los datos escalados y evaluamos su rendimiento.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import randint

# Inicializar el modelo Random Forest
rf = RandomForestClassifier()

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'n_estimators': randint(low=100, high=300),
    'max_depth': randint(low=10, high=30)
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_rf_oversampling = RandomizedSearchCV(
    estimator=rf,                    # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento
random_search_rf_oversampling.fit(X_train, y_train)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros:", random_search_rf_oversampling.best_params_)

# Entrenar el modelo Random Forest con los mejores par?metros
best_rf = random_search_rf_oversampling.best_estimator_

# Predecir con el modelo entrenado en los datos de prueba escalados
y_pred_rf_oversampling = best_rf.predict(X_test)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_test, y_pred_rf_oversampling))
print(classification_report(y_test, y_pred_rf_oversampling))

## An?lisis de Resultados - Random Forest con Oversampling

El modelo Random Forest con oversampling obtuvo una precisi?n general de 94.52%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.0. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo XGBoost

En esta secci?n, entrenaremos un modelo XGBoost para predecir la variable objetivo `stroke`. XGBoost.

Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

A continuaci?n, entrenamos el modelo XGBoost utilizando los datos escalados y evaluamos su rendimiento.

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Crear el clasificador XGBoost
xgb_reg = xgb.XGBClassifier()  # para tareas de clasificaci?n binaria

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'n_estimators': randint(low=100, high=300),
    'max_depth': randint(low=10, high=30),
    'learning_rate': uniform(0.1, 0.2)
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_xgb_oversampling = RandomizedSearchCV(
    estimator=xgb_reg,               # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento
random_search_xgb_oversampling.fit(X_train, y_train)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros:", random_search_xgb_oversampling.best_params_)

# Entrenar el modelo XGBoost con los mejores par?metros
best_xgb = random_search_xgb_oversampling.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba escalados
y_pred_xgb_oversampling = best_xgb.predict(X_test)

# Calcular m?tricas de clasificaci?n con el mejor modelo
print("Accuracy con mejores par?metros:", accuracy_score(y_test, y_pred_xgb_oversampling))
print("\nClassification Report con mejores par?metros:\n", classification_report(y_test, y_pred_xgb_oversampling))
print("\nConfusion Matrix con mejores par?metros:\n", confusion_matrix(y_test, y_pred_xgb_oversampling))

## An?lisis de Resultados - XGBoost con Oversampling

El modelo XGBoost con oversampling obtuvo una precisi?n general de 92.66%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.12. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo Decision Tree con Oversampling

En esta secci?n, entrenaremos y evaluaremos un modelo Decision Tree utilizando los datos balanceados con oversampling. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint

# Crear el modelo Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'max_depth': randint(low=1, high=30),
    'min_samples_split': randint(low=2, high=20),
    'min_samples_leaf': randint(low=1, high=20)
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_dt_oversampling = RandomizedSearchCV(
    estimator=dt,                    # Tu modelo base
    param_distributions=param_distributions,
    n_iter=20,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento balanceados con oversampling
random_search_dt_oversampling.fit(X_train, y_train)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_dt_oversampling.best_params_)

# Entrenar el modelo Decision Tree con los mejores par?metros
best_dt_oversampling = random_search_dt_oversampling.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba
y_pred_dt_oversampling = best_dt_oversampling.predict(X_test)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_test, y_pred_dt_oversampling))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt_oversampling))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt_oversampling))

## An?lisis de Resultados - Decision Tree con Oversampling

El modelo Decision Tree con oversampling obtuvo una precisi?n general de 91.19%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.18. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Conclusi?n de los 4 Modelos Utilizados con Oversampling y Scaling

Aunque los modelos mostraron un buen rendimiento general, su capacidad para predecir la clase minoritaria (Stroke) fue limitada. Esto se refleja en la baja precisi?n, recall y F1-score para la clase 1 (Stroke) en todos los modelos evaluados.

### Pr?ximo Paso

Para mejorar el rendimiento en la clase minoritaria, se pueden considerar las siguientes estrategias:
- **Ajustar el umbral de decisi?n**: Modificar el umbral de probabilidad para clasificar una instancia como clase 1.
- **T?cnicas de balanceo de clases**: Utilizar t?cnicas como SMOTE (Synthetic Minority Over-sampling Technique) para generar m?s ejemplos sint?ticos de la clase minoritaria.

En resumen, aunque los modelos evaluados tienen un buen rendimiento general, es necesario abordar el desbalanceo de clases para mejorar la predicci?n de la clase minoritaria (Stroke). El pr?ximo paso ser? aplicar SMOTE para mejorar el rendimiento en la clase minoritaria.

## Aplicaci?n de SMOTE y Escalado

En esta secci?n, aplicaremos SMOTE (Synthetic Minority Over-sampling Technique) para balancear las clases en el conjunto de entrenamiento. SMOTE genera ejemplos sint?ticos de la clase minoritaria para equilibrar la proporci?n de clases.

Despu?s de aplicar SMOTE, escalaremos los datos utilizando `MinMaxScaler` para asegurar que todas las caracter?sticas est?n en la misma escala.

A continuaci?n, mostramos el c?digo para aplicar SMOTE y escalar los datos.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE

# Dividir los datos en conjuntos de entrenamiento y prueba
X_trainsmote, X_testsmote, y_trainsmote, y_testsmote = train_test_split(features, target, test_size=0.20, stratify=target, random_state=42)

# Agregar la columna 'stroke' a X_trainsmote para facilitar el oversampling
X_trainsmote["stroke"] = y_trainsmote

# Separar las clases mayoritaria y minoritaria
strokesm1 = X_trainsmote[X_trainsmote["stroke"] == 1]
strokesm2 = X_trainsmote[X_trainsmote["stroke"] == 0]

# Combinar las clases mayoritaria y minoritaria en un solo DataFrame
strokesm_combined = pd.concat([strokesm1, strokesm2])

# Separar caracter?sticas y objetivo
X = strokesm_combined.drop(columns=['stroke'])
y = strokesm_combined['stroke']

# Aplicar SMOTE para balancear las clases en el conjunto de entrenamiento
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Escalar los datos utilizando MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_resampled)
X_test_scaled = scaler.transform(X_testsmote)

# Asignar los datos escalados a las variables de entrenamiento y prueba
X_trainsmote = X_train_scaled
y_trainsmote = y_resampled
X_testsmote = X_test_scaled

In [ ]:
# Check the new distribution of the target variable
y_resampled.value_counts().plot(kind="bar")
plt.title("Distribution of 'Stroke' after SMOTE")
plt.show()



## Entrenamiento y Evaluaci?n del Modelo K-Nearest Neighbors (KNN) con SMOTE

En esta secci?n, entrenaremos y evaluaremos un modelo K-Nearest Neighbors (KNN) utilizando los datos balanceados con SMOTE. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint

# Crear el modelo KNN
knn = KNeighborsClassifier()

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'n_neighbors': randint(low=1, high=50),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_knn_smote = RandomizedSearchCV(
    estimator=knn,                   # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento balanceados con SMOTE
random_search_knn_smote.fit(X_trainsmote, y_trainsmote)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_knn_smote.best_params_)

# Entrenar el modelo KNN con los mejores par?metros
best_knn_smote = random_search_knn_smote.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba
y_pred_knn_smote = best_knn_smote.predict(X_testsmote)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_testsmote, y_pred_knn_smote))
print("\nClassification Report:\n", classification_report(y_testsmote, y_pred_knn_smote))
print("\nConfusion Matrix:\n", confusion_matrix(y_testsmote, y_pred_knn_smote))

## An?lisis de Resultados - K-Nearest Neighbors (KNN) con SMOTE

El modelo K-Nearest Neighbors (KNN) con SMOTE obtuvo una precisi?n general de 88.64%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.13. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo de Regresi?n Log?stica con SMOTE

En esta secci?n, entrenaremos y evaluaremos un modelo de Regresi?n Log?stica utilizando los datos balanceados con SMOTE. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.o de Regresi?n Log?stica utilizando los datos escalados y balanceados con SMOTE, y evaluamos su rendimiento.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import uniform

# Crear el modelo de Regresi?n Log?stica
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'C': uniform(loc=0, scale=4),  # Regularizaci?n
    'penalty': ['l2'],             # Tipo de penalizaci?n compatible con lbfgs
    'solver': ['lbfgs', 'liblinear']  # Solvers compatibles con l2
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_log_reg_smote = RandomizedSearchCV(
    estimator=log_reg,               # Tu modelo base
    param_distributions=param_distributions,
    n_iter=20,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento balanceados con SMOTE
random_search_log_reg_smote.fit(X_trainsmote, y_trainsmote)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_log_reg_smote.best_params_)

# Entrenar el modelo de Regresi?n Log?stica con los mejores par?metros
best_log_reg_smote = random_search_log_reg_smote.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba
y_pred_log_reg_smote = best_log_reg_smote.predict(X_testsmote)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_testsmote, y_pred_log_reg_smote))
print("\nClassification Report:\n", classification_report(y_testsmote, y_pred_log_reg_smote))
print("\nConfusion Matrix:\n", confusion_matrix(y_testsmote, y_pred_log_reg_smote))

## An?lisis de Resultados - Regresi?n Log?stica con SMOTE

El modelo de Regresi?n Log?stica con SMOTE obtuvo una precisi?n general de 84.44%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.25. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo XGBoost

En esta secci?n, entrenaremos y evaluaremos un modelo XGBoost utilizando los datos escalados y balanceados con SMOTE. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `GridSearchCV` y `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Crear el clasificador XGBoost
xgb_cla = xgb.XGBClassifier()  


# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'n_estimators': randint(low=100, high=300),
    'max_depth': randint(low=10, high=30),
    'learning_rate': uniform(0.1, 0.2),
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_xgb_smote = RandomizedSearchCV(
    estimator=xgb_cla,               # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento
random_search_xgb_smote.fit(X_trainsmote, y_trainsmote)
print("Mejores par?metros (RandomizedSearchCV):", random_search_xgb_smote.best_params_)

# Entrenar el modelo Random Forest con los mejores par?metros
best_xgb_smote = random_search_xgb_smote.best_estimator_

# Predecir con el modelo entrenado con los mejores par?metros
y_pred_xgb_smote = random_search_xgb_smote.predict(X_testsmote)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_testsmote, y_pred_xgb_smote))
print(classification_report(y_testsmote, y_pred_xgb_smote))

## An?lisis de Resultados - XGBoost con SMOTE

El modelo XGBoost con SMOTE obtuvo una precisi?n general de 92.07%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.16. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo Random Forest con SMOTE

En esta secci?n, entrenaremos y evaluaremos un modelo Random Forest utilizando los datos escalados y balanceados con SMOTE. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `GridSearchCV` y `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.metrics import classification_report, accuracy_score

# Crear el modelo Random Forest con par?metros iniciales
rf = RandomForestClassifier()

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros con RandomizedSearchCV
param_distribs = {
    'n_estimators': randint(low=100, high=300),
    'max_depth': randint(low=10, high=30)
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_rf_smote = RandomizedSearchCV(
    estimator=rf,                    # Tu modelo base
    param_distributions=param_distribs,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento
random_search_rf_smote.fit(X_trainsmote, y_trainsmote)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_rf_smote.best_params_)

# Entrenar el modelo Random Forest con los mejores par?metros
best_rf_smote = random_search_rf_smote.best_estimator_

# Predecir con el modelo entrenado en los datos de prueba escalados
y_pred_rf_smote = best_rf.predict(X_testsmote)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_testsmote, y_pred_rf_smote))
print(classification_report(y_testsmote, y_pred_rf_smote))

## An?lisis de Resultados - Random Forest con SMOTE

El modelo Random Forest con SMOTE obtuvo una precisi?n general de 91.38%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.12. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## Entrenamiento y Evaluaci?n del Modelo Decision Tree con SMOTE

En esta secci?n, entrenaremos y evaluaremos un modelo Decision Tree utilizando los datos balanceados con SMOTE. Tambi?n realizaremos una b?squeda de hiperpar?metros utilizando `RandomizedSearchCV` para encontrar los mejores par?metros para nuestro modelo.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint

# Crear el modelo Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# Definir la distribuci?n de par?metros para la b?squeda de hiperpar?metros
param_distributions = {
    'max_depth': randint(low=1, high=30),
    'min_samples_split': randint(low=2, high=20),
    'min_samples_leaf': randint(low=1, high=20)
}

# Realizar la b?squeda de hiperpar?metros con RandomizedSearchCV
random_search_dt_smote = RandomizedSearchCV(
    estimator=dt,                    # Tu modelo base
    param_distributions=param_distributions,
    n_iter=50,                       # N?mero de combinaciones a probar
    cv=5,                            # N?mero de folds en la validaci?n cruzada
    scoring='f1',                    # M?trica a optimizar
    n_jobs=-1,                       # Usar todos los n?cleos disponibles
    random_state=42,                 # Para reproducibilidad
    verbose=2                        # Para ver el progreso
)

# Ajustar el modelo con los datos de entrenamiento balanceados con SMOTE
random_search_dt_smote.fit(X_trainsmote, y_trainsmote)

# Imprimir los mejores par?metros encontrados
print("Mejores par?metros (RandomizedSearchCV):", random_search_dt_smote.best_params_)

# Entrenar el modelo Decision Tree con los mejores par?metros
best_dt_smote = random_search_dt_smote.best_estimator_

# Realizar predicciones con el modelo entrenado en los datos de prueba
y_pred_dt_smote = best_dt_smote.predict(X_testsmote)

# Calcular precisi?n y otros resultados
print("Accuracy:", accuracy_score(y_testsmote, y_pred_dt_smote))
print("\nClassification Report:\n", classification_report(y_testsmote, y_pred_dt_smote))
print("\nConfusion Matrix:\n", confusion_matrix(y_testsmote, y_pred_dt_smote))

## An?lisis de Resultados - Decision Tree con SMOTE

El modelo Decision Tree con SMOTE obtuvo una precisi?n general de 87.76%. Sin embargo, el rendimiento en la clase minoritaria (Stroke) fue bajo, con un F1-Score de 0.11. Esto indica que el modelo tiene dificultades para predecir correctamente la clase minoritaria, ya que el F1-Score es una m?trica que considera tanto la precisi?n como el recall, y un valor bajo sugiere que el modelo no est? equilibrando bien estas dos m?tricas para la clase minoritaria.

## M?tricas de Rendimiento de Todos los Modelos

A continuaci?n, se presentan las m?tricas de rendimiento (precisi?n, recall, F1-score) para cada uno de los modelos evaluados:


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Diccionario para almacenar los resultados de cada modelo
model_results = {}

# Almacenar los resultados de cada modelo con oversampling
model_results['KNN (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_knn_oversampling),
    'Precision': precision_score(y_test, y_pred_knn_oversampling),
    'Recall': recall_score(y_test, y_pred_knn_oversampling),
    'F1-Score': f1_score(y_test, y_pred_knn_oversampling)
}

model_results['Random Forest (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_rf_oversampling),
    'Precision': precision_score(y_test, y_pred_rf_oversampling),
    'Recall': recall_score(y_test, y_pred_rf_oversampling),
    'F1-Score': f1_score(y_test, y_pred_rf_oversampling)
}

model_results['XGBoost (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_xgb_oversampling),
    'Precision': precision_score(y_test, y_pred_xgb_oversampling),
    'Recall': recall_score(y_test, y_pred_xgb_oversampling),
    'F1-Score': f1_score(y_test, y_pred_xgb_oversampling)
}

model_results['Decision Tree (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_dt_oversampling),
    'Precision': precision_score(y_test, y_pred_dt_oversampling),
    'Recall': recall_score(y_test, y_pred_dt_oversampling),
    'F1-Score': f1_score(y_test, y_pred_dt_oversampling),
}

model_results['Logistic Regression (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_log_reg_oversampling),
    'Precision': precision_score(y_test, y_pred_log_reg_oversampling),
    'Recall': recall_score(y_test, y_pred_log_reg_oversampling),
    'F1-Score': f1_score(y_test, y_pred_log_reg_oversampling)
}

# Almacenar los resultados de cada modelo con SMOTE
model_results['KNN (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_knn_smote),
    'Precision': precision_score(y_testsmote, y_pred_knn_smote),
    'Recall': recall_score(y_testsmote, y_pred_knn_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_knn_smote)
}

model_results['Random Forest (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_rf_smote),
    'Precision': precision_score(y_testsmote, y_pred_rf_smote),
    'Recall': recall_score(y_testsmote, y_pred_rf_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_rf_smote)
}

model_results['XGBoost (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_xgb_smote),
    'Precision': precision_score(y_testsmote, y_pred_xgb_smote),
    'Recall': recall_score(y_testsmote, y_pred_xgb_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_xgb_smote)
}

model_results['Decision Tree (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_dt_smote),
    'Precision': precision_score(y_testsmote, y_pred_dt_smote),
    'Recall': recall_score(y_testsmote, y_pred_dt_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_dt_smote)
}

model_results['Logistic Regression (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_log_reg_smote),
    'Precision': precision_score(y_testsmote, y_pred_log_reg_smote),
    'Recall': recall_score(y_testsmote, y_pred_log_reg_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_log_reg_smote)
}

# Convertir los resultados del diccionario a un DataFrame
results_df = pd.DataFrame(model_results).T.reset_index().rename(columns={'index': 'Model'})

# Transformar los datos para mejor visualizaci?n
results_long = pd.melt(results_df, id_vars='Model', var_name='Metric', value_name='Score')

# Crear el gr?fico comparativo
plt.figure(figsize=(14, 8))
sns.barplot(x='Model', y='Score', hue='Metric', data=results_long)
plt.title('Comparaci?n de Modelos con Oversampling y SMOTE')
plt.ylabel('Score')
plt.xlabel('Model')
plt.legend(loc='best')
plt.xticks(rotation=45)
plt.show()

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Diccionario para almacenar los resultados de cada modelo
model_results = {}

# Almacenar los resultados de cada modelo con oversampling
model_results['KNN (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_knn_oversampling),
    'Precision': precision_score(y_test, y_pred_knn_oversampling),
    'Recall': recall_score(y_test, y_pred_knn_oversampling),
    'F1-Score': f1_score(y_test, y_pred_knn_oversampling)
}

model_results['Random Forest (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_rf_oversampling),
    'Precision': precision_score(y_test, y_pred_rf_oversampling),
    'Recall': recall_score(y_test, y_pred_rf_oversampling),
    'F1-Score': f1_score(y_test, y_pred_rf_oversampling)
}

model_results['XGBoost (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_xgb_oversampling),
    'Precision': precision_score(y_test, y_pred_xgb_oversampling),
    'Recall': recall_score(y_test, y_pred_xgb_oversampling),
    'F1-Score': f1_score(y_test, y_pred_xgb_oversampling)
}

model_results['Decision Tree (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_dt_oversampling),
    'Precision': precision_score(y_test, y_pred_dt_oversampling),
    'Recall': recall_score(y_test, y_pred_dt_oversampling),
    'F1-Score': f1_score(y_test, y_pred_dt_oversampling),
}

model_results['Logistic Regression (Oversampling)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_log_reg_oversampling),
    'Precision': precision_score(y_test, y_pred_log_reg_oversampling),
    'Recall': recall_score(y_test, y_pred_log_reg_oversampling),
    'F1-Score': f1_score(y_test, y_pred_log_reg_oversampling)
}

# Almacenar los resultados de cada modelo con SMOTE
model_results['KNN (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_knn_smote),
    'Precision': precision_score(y_testsmote, y_pred_knn_smote),
    'Recall': recall_score(y_testsmote, y_pred_knn_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_knn_smote)
}

model_results['Random Forest (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_rf_smote),
    'Precision': precision_score(y_testsmote, y_pred_rf_smote),
    'Recall': recall_score(y_testsmote, y_pred_rf_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_rf_smote)
}

model_results['XGBoost (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_xgb_smote),
    'Precision': precision_score(y_testsmote, y_pred_xgb_smote),
    'Recall': recall_score(y_testsmote, y_pred_xgb_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_xgb_smote)
}

model_results['Decision Tree (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_dt_smote),
    'Precision': precision_score(y_testsmote, y_pred_dt_smote),
    'Recall': recall_score(y_testsmote, y_pred_dt_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_dt_smote)
}

model_results['Logistic Regression (SMOTE)'] = {
    'Accuracy': accuracy_score(y_testsmote, y_pred_log_reg_smote),
    'Precision': precision_score(y_testsmote, y_pred_log_reg_smote),
    'Recall': recall_score(y_testsmote, y_pred_log_reg_smote),
    'F1-Score': f1_score(y_testsmote, y_pred_log_reg_smote)
}

# Convertir los resultados del diccionario a un DataFrame
results_df = pd.DataFrame(model_results).T.reset_index().rename(columns={'index': 'Model'})

# Mostrar el DataFrame
results_df

## Conclusi?n Final


### Mejor Modelo

El modelo **Logistic Regression (SMOTE)** parece ser el mejor modelo en t?rminos de balance entre precisi?n, recall y F1-Score. Aunque **Random Forest (SMOTE)** y **XGBoost (SMOTE)** tienen una mayor precisi?n general, **Logistic Regression (SMOTE)** ofrece un mejor recall y F1-Score, lo que lo hace m?s efectivo para identificar la clase minoritaria (Stroke).


### Pr?ximos Pasos

Para mejorar el rendimiento en la clase minoritaria, se pueden considerar las siguientes estrategias:
- **Ajustar el umbral de decisi?n**: Modificar el umbral de probabilidad para clasificar una instancia como clase 1.
- **Modelos m?s avanzados**: Probar modelos m?s avanzados como redes neuronales profundas o ensamblajes de modelos.

En resumen, aunque los modelos evaluados tienen un buen rendimiento general, es necesario abordar el desbalanceo de clases para mejorar la predicci?n de la clase minoritaria (Stroke). Debido a esta falta de consistencia en la predicci?n de la clase minoritaria, no recomendamos continuar con este proyecto ni llevarlo a producci?n en su estado actual. Sin embargo, continuaremos trabajando en este proyecto con fines educativos, explorando nuevas t?cnicas y enfoques para mejorar el rendimiento de los modelos.

## Entrenamiento y Preparaci?n del Modelo Final para Producci?n

En esta secci?n, seleccionaremos el mejor modelo basado en los resultados anteriores, entrenaremos el modelo con todos los datos disponibles y prepararemos el modelo para su despliegue en producci?n.

### Selecci?n del Mejor Modelo

Basado en los resultados anteriores, seleccionamos el modelo **Logistic Regression (SMOTE)** como el mejor modelo debido a su balance entre precisi?n, recall y F1-Score.

### Entrenamiento del Modelo Final

Entrenaremos el modelo **Logistic Regression (SMOTE)** utilizando todos los datos disponibles y los mejores hiperpar?metros encontrados durante la b?squeda de hiperpar?metros.

In [ ]:
X

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE



# Usar las variables preprocesadas directamente
X = df.drop(columns=["stroke"])  # Caracter?sticas preprocesadas
y = df["stroke"]    # Etiquetas preprocesadas

#Se hace el escalado y se guarda el pkl
# Escalar los datos utilizando MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

#Guardar escalado
joblib.dump(scaler, 'scalerfinal.pkl')

# Agregar la columna 'stroke' a X para facilitar el SMOTE
X["stroke"] = y

# Separar las clases mayoritaria y minoritaria
strokefinal1 = X[X["stroke"] == 1]
strokefinal2 = X[X["stroke"] == 0]

# Combinar las clases mayoritaria y minoritaria en un solo DataFrame
strokefinal_combined = pd.concat([strokefinal1, strokefinal2])

# Separar caracter?sticas y objetivo
X = strokefinal_combined.drop(columns=['stroke'])
y = strokefinal_combined['stroke']


# Aplicar SMOTE para balancear las clases
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

#GUARDAR SMOTE PKL
joblib.dump(smote, 'smotefinal.pkl')

# Crear el modelo Logistic Regression con los mejores hiperpar?metros
best_params = {'C': 1.8670515729919197, 'penalty': 'l2', 'solver': 'lbfgs'} 
model = LogisticRegression(**best_params)

# Entrenar el modelo con todos los datos disponibles
model.fit(X_smote, y_smote)

# Guardar el modelo entrenado para su despliegue
joblib.dump(model, 'logistic_regression_smote_model.pkl')

### Preparaci?n del Modelo para Despliegue

El modelo entrenado se ha guardado en un archivo `logistic_regression_smote_model.pkl` utilizando la biblioteca `joblib`. Este archivo puede ser cargado y utilizado para realizar predicciones en un entorno de producci?n.

### C?digo para Cargar y Utilizar el Modelo en Producci?n

A continuaci?n se muestra un ejemplo de c?mo cargar el modelo guardado y utilizarlo para realizar predicciones en nuevos datos.

In [ ]:
import joblib
import pandas as pd

# Cargar el modelo guardado
model = joblib.load('random_forest_smote_model.pkl')

# Funci?n para solicitar los datos del paciente
def get_patient_data():
    data = {}
    data['age'] = float(input("Ingrese la edad del paciente: "))
    data['hypertension'] = int(input("?El paciente tiene hipertensi?n? (1 para S?, 0 para No): "))
    data['heart_disease'] = int(input("?El paciente tiene enfermedad card?aca? (1 para S?, 0 para No): "))
    data['avg_glucose_level'] = float(input("Ingrese el nivel promedio de glucosa del paciente: "))
    data['bmi'] = float(input("Ingrese el ?ndice de masa corporal (BMI) del paciente: "))
    data['gender_Male'] = int(input("?El paciente es masculino? (1 para S?, 0 para No): "))
    data['ever_married_Yes'] = int(input("?El paciente alguna vez se ha casado? (1 para S?, 0 para No): "))
    data['work_type_Never_worked'] = int(input("?El paciente nunca ha trabajado? (1 para S?, 0 para No): "))
    data['work_type_Private'] = int(input("?El paciente trabaja en el sector privado? (1 para S?, 0 para No): "))
    data['work_type_Self-employed'] = int(input("?El paciente es aut?nomo? (1 para S?, 0 para No): "))
    data['work_type_children'] = int(input("?El paciente es un ni?o? (1 para S?, 0 para No): "))
    data['Residence_type_Urban'] = int(input("?El paciente vive en una zona urbana? (1 para S?, 0 para No): "))
    data['smoking_status_formerly smoked'] = int(input("?El paciente fumaba anteriormente? (1 para S?, 0 para No): "))
    data['smoking_status_never smoked'] = int(input("?El paciente nunca ha fumado? (1 para S?, 0 para No): "))
    data['smoking_status_smokes'] = int(input("?El paciente fuma actualmente? (1 para S?, 0 para No): "))
    
    return pd.DataFrame([data])

# Solicitar los datos del paciente
patient_data = get_patient_data()

# Realizar predicciones con el modelo cargado
prediction = model.predict(patient_data)

# Mostrar las predicciones
if prediction[0] == 1:
    print("El paciente tiene un riesgo de sufrir un accidente cerebrovascular.")
else:
    print("El paciente no tiene un riesgo significativo de sufrir un accidente cerebrovascular.")